# Breast Cancer Wisconsin — Diagnostic
## Machine Learning Project for Medical Diagnosis

**Dataset:** [UCI — Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)

**Objective:** Classify breast tumors as **benign (B)** or **malignant (M)** based on 30 numerical features extracted from Fine Needle Aspirate (FNA) images.

**Dataset contents:**
- **569 observations** (212 malignant, 357 benign)
- **30 features** (mean, standard error, worst for: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension)

**Project pipeline:**
1. Data loading and exploration
2. Correlation analysis
3. Dimensionality reduction (PCA)
4. 3D visualization (PCA)
5. Discriminant analysis (LDA / FDA)
6. SVM (Support Vector Machine)
7. MLP (Neural Network — Multi-Layer Perceptron)

## 1. Import Libraries

We import the necessary libraries:
- **pandas** for data manipulation
- **matplotlib / seaborn** for visualization
- **scikit-learn** for ML models (PCA, LDA, SVM, MLP), preprocessing and evaluation metrics

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, make_scorer
)

# Plot style
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Data Loading

We load the `data.csv` file containing the Wisconsin Breast Cancer Diagnostic dataset.
The file has no header, so we define the column names manually.
The separator (comma or semicolon) is detected automatically.

In [ ]:
columns = [
    "id", "diagnosis",
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
    "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
    "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

csv_path = Path("data.csv")

# Automatic separator detection
with open(str(csv_path), "r", encoding="utf-8") as f:
    first_line = f.readline()
    sep = ',' if first_line.count(',') >= first_line.count(';') else ';'

df = pd.read_csv(str(csv_path), header=None, names=columns, sep=sep)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Detected separator: '{sep}'")

## 3. Data Exploration

### 3.1 First and Last Rows

We display the first and last rows of the DataFrame for a quick overview of the data structure.

In [ ]:
print("=== First rows ===")
display(df.head())

print("\n=== Last rows ===")
display(df.tail())

### 3.2 General Information

`df.info()` shows the number of rows, columns, data types and non-null counts per column.

In [ ]:
print(f"DataFrame dimensions: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.info()

### 3.3 Descriptive Statistics

`df.describe()` provides basic statistics (mean, standard deviation, min, max, quartiles) for numerical columns.

In [ ]:
display(df.describe())

# Statistics for the categorical column (diagnosis)
print("\n=== Categorical column ===")
display(df.describe(include='object'))

### 3.4 Diagnosis Distribution

We check the balance between Benign (B) and Malignant (M) classes. A significant imbalance could require resampling techniques.

In [ ]:
counts = df['diagnosis'].value_counts()
print(f"Healthy patients (B — Benign): {counts.get('B', 0)}")
print(f"Sick patients (M — Malignant): {counts.get('M', 0)}")
print(f"M/B ratio: {counts.get('M', 0) / counts.get('B', 0):.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', color=['green', 'red'], edgecolor='black', ax=ax)
ax.set_title("Diagnosis Distribution")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of patients")
ax.set_xticklabels(['Benign (B)', 'Malignant (M)'], rotation=0)
plt.tight_layout()
plt.show()

### 3.5 Missing Values

We check for missing values in the dataset. This is an essential step before any modeling.

In [ ]:
missing = df.isnull().sum()
total_missing = missing.sum()
print(f"Total missing values: {total_missing}")

if total_missing > 0:
    print("\nMissing values per column:")
    display(missing[missing > 0])
else:
    print("No missing values in the dataset.")

### 3.6 Data Types and Unique Values

We examine the data types of each column and the number of unique values to detect potential anomalies.

In [ ]:
types_df = pd.DataFrame({
    'Type': df.dtypes,
    'Unique values': df.nunique(),
    'Example': df.iloc[0]
})
display(types_df)

### 3.7 Numerical Feature Distributions

We visualize the distribution of each numerical variable using histograms. This helps identify skewed distributions, outliers, or variables that may require transformation.

In [ ]:
numeric_cols = df.select_dtypes(include='number').drop(columns=['id'])
fig, axes = plt.subplots(6, 5, figsize=(20, 18))
axes = axes.flatten()

for i, col in enumerate(numeric_cols.columns):
    numeric_cols[col].hist(bins=25, ax=axes[i], color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)

# Hide empty axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution of the 30 Numerical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

We encode the diagnosis (B=0, M=1) and compute Pearson correlations to find the most discriminative features.

Attributes with high absolute correlation are considered more discriminative, as they tend to vary significantly between benign and malignant tumors, and are therefore potentially good indicators for diagnosis.

In [ ]:
# Encode diagnosis: B → 0, M → 1
df['diagnosis_encoded'] = df['diagnosis'].map({'B': 0, 'M': 1})

# Correlation of each feature with the diagnosis
correlations = df.drop(columns=['id', 'diagnosis']).corr()['diagnosis_encoded'].drop('diagnosis_encoded')
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("Top 10 attributes most correlated with diagnosis (absolute value):\n")
display(correlations_sorted.head(10))

# Bar chart
top_features = correlations_sorted.head(10)
plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='mediumseagreen')
plt.title("Top 10 Attributes Most Correlated with Diagnosis (M vs B)")
plt.xlabel("Correlation Coefficient (absolute value)")
plt.gca().invert_yaxis()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Full Correlation Matrix (Heatmap)

The heatmap below shows the pairwise correlations between all numerical features. This helps identify groups of highly correlated variables (multicollinearity).

In [ ]:
corr_matrix = df.drop(columns=['id', 'diagnosis']).corr()

plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, linewidths=0.5)
plt.title("Correlation Matrix of Numerical Features", fontsize=14)
plt.tight_layout()
plt.show()

## 5. PCA — Principal Component Analysis

PCA is an **unsupervised** dimensionality reduction method. It projects the data into a lower-dimensional space while retaining as much variance as possible.

**Steps:**
1. Standardize the data (essential since PCA is sensitive to feature scales)
2. Compute the principal components
3. Analyze cumulative explained variance to choose the number of components
4. Apply the dimensionality reduction

In [ ]:
# Select numerical features (excluding id, diagnosis, diagnosis_encoded)
features = df.drop(columns=['id', 'diagnosis', 'diagnosis_encoded'])

# Standardization
scaler_pca = StandardScaler()
scaled_features = scaler_pca.fit_transform(features)

# Full PCA (fit only — we just need the explained variance ratios)
pca = PCA()
pca.fit(scaled_features)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = explained_variance_ratio.cumsum()

# Cumulative explained variance plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_variance, marker='o', linestyle='--')
plt.axhline(y=0.95, color='r', linestyle='-', label='95% explained variance')
plt.title('Cumulative Explained Variance by Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.show()

# Number of components needed to explain 95% of the variance
n_components = next(i for i, v in enumerate(cumulative_variance) if v >= 0.95) + 1
print(f"\nNumber of components needed for 95% variance: {n_components}")

# Apply PCA with the optimal number of components
pca_final = PCA(n_components=n_components)
principal_components_final = pca_final.fit_transform(scaled_features)
print(f"Data dimensions after PCA: {principal_components_final.shape}")

### 3D PCA Visualization

We project the data into 3D space (first 3 principal components) to visually inspect class separability. Each point represents one observation:
- **Green** = Benign (B)
- **Red** = Malignant (M)

In [ ]:
# PCA with 3 components for 3D visualization
pca_3d = PCA(n_components=3)
pc_3d = pca_3d.fit_transform(scaled_features)

pca_df_3d = pd.DataFrame(pc_3d, columns=['PC1', 'PC2', 'PC3'])
pca_df_3d['diagnosis'] = df['diagnosis']

# 3D Visualization
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(pca_df_3d['PC1'], pca_df_3d['PC2'], pca_df_3d['PC3'],
           c=pca_df_3d['diagnosis'].map({'B': 'green', 'M': 'red'}), alpha=0.6)

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
ax.set_title('3D PCA Visualization — Breast Cancer Dataset')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, label='Benign'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='Malignant')
]
ax.legend(handles=legend_elements, title="Diagnosis")
plt.show()

## 6. Train / Test Split

Before training models, we split the data into a **training set** (80%) and a **test set** (20%).

We use the **same split** for all models to ensure a fair comparison:
- `test_size=0.2` → 20% of data for testing
- `random_state=42` → reproducibility
- `stratify=y` → same B/M proportions in train and test

In [ ]:
# Prepare data
X = df.drop(columns=['id', 'diagnosis', 'diagnosis_encoded'])
y = df['diagnosis_encoded']

# Single consistent split for all models
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTrain distribution: B={(y_train == 0).sum()}, M={(y_train == 1).sum()}")
print(f"Test distribution:  B={(y_test == 0).sum()}, M={(y_test == 1).sum()}")

## 7. LDA — Linear Discriminant Analysis (FDA)

Linear Discriminant Analysis (LDA, also known as FDA — Fisher Discriminant Analysis) is a **supervised** method that projects the data onto an axis that **maximizes class separation**.

Unlike PCA (unsupervised), LDA uses class labels to find the optimal projection.

With 2 classes (B and M), LDA reduces the data to **1 single dimension**. We then evaluate the discriminative power of this projection using logistic regression.

In [ ]:
# Standardization
scaler_lda = StandardScaler()
X_train_lda_scaled = scaler_lda.fit_transform(X_train)
X_test_lda_scaled = scaler_lda.transform(X_test)

# Apply LDA (reduce to 1 component)
lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_lda_scaled, y_train)
X_test_lda = lda.transform(X_test_lda_scaled)

print(f"Dimensions after LDA — Train: {X_train_lda.shape}, Test: {X_test_lda.shape}")

# Visualize class separation on the discriminant axis
plt.figure(figsize=(10, 4))
plt.scatter(X_train_lda[y_train == 0], [0] * (y_train == 0).sum(), label='Benign', color='green', alpha=0.7)
plt.scatter(X_train_lda[y_train == 1], [0] * (y_train == 1).sum(), label='Malignant', color='red', alpha=0.7)
plt.xlabel('Linear Discriminant Component 1')
plt.yticks([])
plt.title('Class Separation in LDA Space (Training Data)')
plt.legend()
plt.grid(True)
plt.show()

# Logistic regression on the LDA projection
logistic_regression = LogisticRegression(max_iter=1000)
logistic_regression.fit(X_train_lda, y_train)
y_pred_lda = logistic_regression.predict(X_test_lda)
accuracy_lda = accuracy_score(y_test, y_pred_lda)
print(f"\nLogistic regression accuracy on LDA component: {accuracy_lda:.4f}")
print(f"\nLinear discriminant function coefficients:\n{lda.coef_}")

## 8. SVM — Support Vector Machine

SVM finds the **optimal hyperplane** that separates the two classes with the **maximum margin**.

We use a linear kernel (`kernel='linear'`), which is well-suited when the number of features is high relative to the number of samples.

The 2D visualization is obtained by projecting the data with PCA (fitted only on the training set to avoid data leakage).

In [ ]:
# Standardization
scaler_svm = StandardScaler()
X_train_svm_scaled = scaler_svm.fit_transform(X_train)
X_test_svm_scaled = scaler_svm.transform(X_test)

# Train linear SVM
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_svm_scaled, y_train)

# Predictions and evaluation
y_pred_svm = svm_model.predict(X_test_svm_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"SVM accuracy: {acc_svm:.4f}")
print(f"\nClassification report:\n{classification_report(y_test, y_pred_svm, target_names=['Benign', 'Malignant'])}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred_svm)}")

In [ ]:
# 2D visualization — PCA fitted on training data only (no data leakage)
pca_vis = PCA(n_components=2)
X_train_vis = pca_vis.fit_transform(X_train_svm_scaled)
X_vis = pca_vis.transform(scaler_svm.transform(X))

# Retrain SVM on the 2D projected training data
svm_model_vis = SVC(kernel='linear')
svm_model_vis.fit(X_train_vis, y_train)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y, cmap='bwr', alpha=0.6, edgecolors='k')
plt.title('SVM — Data Projected in 2D (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.legend(*scatter.legend_elements(), title="Diagnosis", loc="best")
plt.tight_layout()
plt.show()

## 9. MLP — Multi-Layer Perceptron (Neural Network)

The MLP is a supervised neural network consisting of fully connected layers of neurons.

**Our approach:**
1. **Standardization** of the data
2. **PCA** for dimensionality reduction (keep 95% variance) — reduces overfitting risk
3. **GridSearchCV** for hyperparameter tuning:
   - Architectures tested: `(50,)`, `(100,)`, `(100, 50)`
   - Regularization (alpha): `0.0001`, `0.001`, `0.01`
   - Learning rate: `constant`, `adaptive`
   - Early stopping enabled to prevent overfitting
4. **5-fold cross-validation** with macro F1-score as the evaluation metric

In [ ]:
# Standardization
scaler_mlp = StandardScaler()
X_train_mlp_scaled = scaler_mlp.fit_transform(X_train)
X_test_mlp_scaled = scaler_mlp.transform(X_test)

# PCA for dimensionality reduction
pca_mlp = PCA(n_components=0.95)
X_train_mlp_pca = pca_mlp.fit_transform(X_train_mlp_scaled)
X_test_mlp_pca = pca_mlp.transform(X_test_mlp_scaled)
print(f"Dimensions after PCA (95% variance): {X_train_mlp_pca.shape[1]} components")

# Hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300],
    'early_stopping': [True],
    'validation_fraction': [0.15],
    'n_iter_no_change': [10]
}

mlp = MLPClassifier(solver='adam', random_state=42)
f1_scorer = make_scorer(f1_score, average='macro')

# Grid search with 5-fold cross-validation
grid = GridSearchCV(mlp, param_grid, cv=5, scoring=f1_scorer, verbose=2, n_jobs=-1)
grid.fit(X_train_mlp_pca, y_train)

print(f"\nBest hyperparameters: {grid.best_params_}")

In [ ]:
target_names = ['Benign', 'Malignant']

# Evaluation on the test set
best_mlp = grid.best_estimator_
y_pred_mlp = best_mlp.predict(X_test_mlp_pca)

print("Final MLP evaluation on the test set:")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_mlp):.4f}")
print(f"  F1-score  : {f1_score(y_test, y_pred_mlp, average='macro'):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_mlp, average='macro'):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_mlp, average='macro'):.4f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_test, y_pred_mlp)}")
print(f"\nClassification report:\n{classification_report(y_test, y_pred_mlp, target_names=target_names)}")

## 10. Model Comparison

Summary of each model's performance evaluated on the **same test set** (20%, stratified, `random_state=42`).

In [ ]:
# Model comparison table
results = pd.DataFrame({
    'Model': ['LDA + Logistic Regression', 'SVM (linear)', 'MLP (neural network)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lda),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_mlp)
    ],
    'F1-score (macro)': [
        f1_score(y_test, y_pred_lda, average='macro'),
        f1_score(y_test, y_pred_svm, average='macro'),
        f1_score(y_test, y_pred_mlp, average='macro')
    ],
    'Recall (macro)': [
        recall_score(y_test, y_pred_lda, average='macro'),
        recall_score(y_test, y_pred_svm, average='macro'),
        recall_score(y_test, y_pred_mlp, average='macro')
    ],
    'Precision (macro)': [
        precision_score(y_test, y_pred_lda, average='macro'),
        precision_score(y_test, y_pred_svm, average='macro'),
        precision_score(y_test, y_pred_mlp, average='macro')
    ]
})

display(results.style.highlight_max(subset=['Accuracy', 'F1-score (macro)', 'Recall (macro)', 'Precision (macro)'],
                                      color='lightgreen'))

# Comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
results.set_index('Model')[['Accuracy', 'F1-score (macro)', 'Recall (macro)', 'Precision (macro)']].plot(
    kind='bar', ax=ax, edgecolor='black'
)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0.85, 1.0)
ax.legend(loc='lower right')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Conclusion

This notebook implements a complete Machine Learning pipeline for breast cancer diagnosis:

1. **Data exploration**: 569 observations, 30 features, no missing values, slightly imbalanced classes (63% B, 37% M)
2. **Correlation analysis**: features `concave_points_worst`, `perimeter_worst`, `radius_worst` are the most discriminative
3. **PCA**: dimensionality reduction from 30 → ~10 components for 95% retained variance
4. **LDA**: good class separation on a single discriminant axis
5. **Linear SVM**: excellent performance with an optimal separating hyperplane
6. **MLP**: neural network with hyperparameter optimization via GridSearchCV

**recall**,**In a medical context**  (sensitivity) is the most important metric because a false negative (malignant tumor classified as benign) has far more severe consequences than a false positive.